In [1]:
%load_ext tensorboard

In [2]:
!rmdir /S /Q logs

In [3]:
from datetime import datetime
from packaging import version
import os

import tensorflow as tf
from tensorflow import keras
from keras import backend as K
import numpy as np

from tensorboard.plugins.hparams import api as hp

print("TensorFlow version:", tf.__version__)
assert version.parse(tf.__version__).release[0] >= 2, \
    "This lab requires TensorFlow 2.0 or above."

# Enable TensorBoard debugger data dumping (shared logs folder)
tf.debugging.experimental.enable_dump_debug_info(
    './logs/debug',
    tensor_debug_mode="FULL_HEALTH",
    circular_buffer_size=-1
)


TensorFlow version: 2.10.0
INFO:tensorflow:Enabled dumping callback in thread MainThread (dump root: ./logs/debug, tensor debug mode: FULL_HEALTH)


In [4]:
def run_regression_demo():
    # Synthetic regression data: y = 0.8x - 1 + noise
    n_samples = 1200
    train_frac = 0.75

    x = np.linspace(-2.0, 2.0, n_samples)
    np.random.shuffle(x)
    y = 0.8 * x - 1.0 + np.random.normal(0, 0.12, size=n_samples)

    split = int(train_frac * n_samples)
    x_train, y_train = x[:split], y[:split]
    x_val, y_val = x[split:], y[split:]

    x_train = x_train[..., np.newaxis]
    x_val   = x_val[..., np.newaxis]

    logdir = "logs/regression/" + datetime.now().strftime("%Y%m%d-%H%M%S")

    tb_callback = keras.callbacks.TensorBoard(
        log_dir=logdir,
        histogram_freq=1,       # log weight histograms
        write_graph=True,
        write_images=False
    )

    model = keras.Sequential([
        keras.layers.Input(shape=(1,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.03),
        loss="mse",
        metrics=["mae"]
    )

    print("Training regression demo model...")
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=25,
        batch_size=64,
        callbacks=[tb_callback],
        verbose=1
    )

    print("Average train loss:", np.mean(history.history["loss"]))
    print("Average val  loss:", np.mean(history.history["val_loss"]))

    return model, (x_train, y_train, x_val, y_val), logdir

reg_model, reg_data, reg_logdir = run_regression_demo()


Training regression demo model...
Epoch 1/25
15/15 [==============================] - 14s 510ms/step - loss: 0.2583 - mae: 0.3409 - val_loss: 0.0319 - val_mae: 0.1515
Epoch 2/25
15/15 [==============================] - 5s 364ms/step - loss: 0.0238 - mae: 0.1239 - val_loss: 0.0241 - val_mae: 0.1221
Epoch 3/25
15/15 [==============================] - 5s 370ms/step - loss: 0.0180 - mae: 0.1080 - val_loss: 0.0125 - val_mae: 0.0885
Epoch 4/25
15/15 [==============================] - 5s 370ms/step - loss: 0.0174 - mae: 0.1049 - val_loss: 0.0157 - val_mae: 0.1002
Epoch 5/25
15/15 [==============================] - 5s 374ms/step - loss: 0.0157 - mae: 0.0992 - val_loss: 0.0121 - val_mae: 0.0871
Epoch 6/25
15/15 [==============================] - 6s 421ms/step - loss: 0.0174 - mae: 0.1050 - val_loss: 0.0149 - val_mae: 0.0965
Epoch 7/25
15/15 [==============================] - 8s 552ms/step - loss: 0.0156 - mae: 0.1006 - val_loss: 0.0145 - val_mae: 0.0945
Epoch 8/25
15/15 [=======================

In [5]:
def load_fashion_mnist():
    (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
    x_train = x_train.astype("float32") / 255.0
    x_test  = x_test.astype("float32") / 255.0

    # Add channel dimension
    x_train = x_train[..., np.newaxis]
    x_test  = x_test[..., np.newaxis]

    return (x_train, y_train), (x_test, y_test)

In [6]:
def build_cnn_model():
    model = keras.Sequential([
        keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [7]:
def run_cnn_with_tensorboard():
    (x_train, y_train), (x_test, y_test) = load_fashion_mnist()

    logdir = "logs/cnn_fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
    tb_callback = keras.callbacks.TensorBoard(
        log_dir=logdir,
        histogram_freq=1,
        write_graph=True
    )

    model = build_cnn_model()
    print("Training CNN classifier...")
    model.fit(
        x_train, y_train,
        epochs=3,
        batch_size=128,
        validation_data=(x_test, y_test),
        callbacks=[tb_callback],
        verbose=1
    )
    return model, ((x_train, y_train), (x_test, y_test)), logdir

cnn_model, cnn_data, cnn_logdir = run_cnn_with_tensorboard()

Training CNN classifier...
Epoch 1/3
469/469 [==============================] - 56s 102ms/step - loss: 0.6192 - accuracy: 0.7758 - val_loss: 0.4294 - val_accuracy: 0.8434
Epoch 2/3
469/469 [==============================] - 50s 107ms/step - loss: 0.3853 - accuracy: 0.8607 - val_loss: 0.3674 - val_accuracy: 0.8609
Epoch 3/3
469/469 [==============================] - 50s 106ms/step - loss: 0.3330 - accuracy: 0.8773 - val_loss: 0.3252 - val_accuracy: 0.8790


In [7]:
def run_profiler_demo():
    (x_train, y_train), (x_test, y_test) = load_fashion_mnist()

    logdir = "logs/profiler/" + datetime.now().strftime("%Y%m%d-%H%M%S")
    profiler_tb = keras.callbacks.TensorBoard(
        log_dir=logdir,
        histogram_freq=0,
        profile_batch="10,20"  # profile only a few batches
    )

    model = build_cnn_model()
    print("Training CNN with profiler enabled...")
    model.fit(
        x_train, y_train,
        epochs=2,
        batch_size=256,
        validation_data=(x_test, y_test),
        callbacks=[profiler_tb],
        verbose=1
    )

    return model, logdir

prof_model, profiler_logdir = run_profiler_demo()

Training CNN with profiler enabled...
Epoch 1/2
235/235 [==============================] - 47s 163ms/step - loss: 0.6833 - accuracy: 0.7509 - val_loss: 0.4435 - val_accuracy: 0.8361
Epoch 2/2
235/235 [==============================] - 32s 138ms/step - loss: 0.4242 - accuracy: 0.8467 - val_loss: 0.3799 - val_accuracy: 0.8636


In [8]:
HP_NUM_UNITS = hp.HParam('num_units', hp.Discrete([64, 128]))
HP_DROPOUT   = hp.HParam('dropout', hp.Discrete([0.2, 0.4]))
HP_OPTIMIZER = hp.HParam('optimizer', hp.Discrete(['adam', 'sgd']))

METRIC_ACCURACY = 'accuracy'

hparams_logdir = 'logs/hparams'

with tf.summary.create_file_writer(hparams_logdir).as_default():
    hp.hparams_config(
        hparams=[HP_NUM_UNITS, HP_DROPOUT, HP_OPTIMIZER],
        metrics=[hp.Metric(METRIC_ACCURACY, display_name='Accuracy')],
    )

In [9]:
def build_hparam_model(hparams):
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(hparams[HP_NUM_UNITS], activation='relu'),
        keras.layers.Dropout(hparams[HP_DROPOUT]),
        keras.layers.Dense(10, activation='softmax')
    ])

    optimizer_name = hparams[HP_OPTIMIZER]
    if optimizer_name == 'adam':
        opt = keras.optimizers.Adam(0.001)
    else:
        opt = keras.optimizers.SGD(0.01, momentum=0.9)

    model.compile(
        optimizer=opt,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [10]:
(x_train_hp, y_train_hp), (x_test_hp, y_test_hp) = keras.datasets.fashion_mnist.load_data()
x_train_hp = (x_train_hp.astype('float32') / 255.0)
x_test_hp  = (x_test_hp.astype('float32') / 255.0)

# To keep it fast, use a subset of the data
x_train_hp_small = x_train_hp[:10000]
y_train_hp_small = y_train_hp[:10000]
x_val_hp_small   = x_test_hp[:2000]
y_val_hp_small   = y_test_hp[:2000]

In [11]:
def run_hparam_trial(run_dir, hparams):
    model = build_hparam_model(hparams)
    with tf.summary.create_file_writer(run_dir).as_default():
        hp.hparams(hparams)   # log hparams
        history = model.fit(
            x_train_hp_small, y_train_hp_small,
            epochs=3,
            batch_size=128,
            validation_data=(x_val_hp_small, y_val_hp_small),
            verbose=0
        )
        _, acc = model.evaluate(x_val_hp_small, y_val_hp_small, verbose=0)
        # Log the metric that HParams dashboard will read
        tf.summary.scalar(METRIC_ACCURACY, acc, step=1)
    return acc

In [12]:
session_num = 0
for num_units in HP_NUM_UNITS.domain.values:
    for dropout_rate in HP_DROPOUT.domain.values:
        for optimizer in HP_OPTIMIZER.domain.values:
            hparams = {
                HP_NUM_UNITS: num_units,
                HP_DROPOUT: dropout_rate,
                HP_OPTIMIZER: optimizer,
            }
            run_name = f"run-{session_num}"
            print('--- Starting trial:', run_name)
            print({h.name: hparams[h] for h in hparams})
            run_dir = os.path.join(hparams_logdir, run_name)
            acc = run_hparam_trial(run_dir, hparams)
            print("Final validation accuracy: %.4f" % acc)
            session_num += 1

--- Starting trial: run-0
{'num_units': 64, 'dropout': 0.2, 'optimizer': 'adam'}


Final validation accuracy: 0.8140
--- Starting trial: run-1
{'num_units': 64, 'dropout': 0.2, 'optimizer': 'sgd'}
Final validation accuracy: 0.7915
--- Starting trial: run-2
{'num_units': 64, 'dropout': 0.4, 'optimizer': 'adam'}
Final validation accuracy: 0.7950
--- Starting trial: run-3
{'num_units': 64, 'dropout': 0.4, 'optimizer': 'sgd'}
Final validation accuracy: 0.7965
--- Starting trial: run-4
{'num_units': 128, 'dropout': 0.2, 'optimizer': 'adam'}
Final validation accuracy: 0.8220
--- Starting trial: run-5
{'num_units': 128, 'dropout': 0.2, 'optimizer': 'sgd'}
Final validation accuracy: 0.7935
--- Starting trial: run-6
{'num_units': 128, 'dropout': 0.4, 'optimizer': 'adam'}
Final validation accuracy: 0.8390
--- Starting trial: run-7
{'num_units': 128, 'dropout': 0.4, 'optimizer': 'sgd'}
Final validation accuracy: 0.8100


In [13]:
%tensorboard --logdir logs/

Reusing TensorBoard on port 6006 (pid 31208), started 2:14:31 ago. (Use '!kill 31208' to kill it.)

: 

: 

: 

: 

: 

: 

: 

In [ ]:
from datetime import datetime
from packaging import version

import tensorflow as tf
from tensorflow import keras
tf.debugging.experimental.enable_dump_debug_info('./logs/',
                                                 tensor_debug_mode="FULL_HEALTH", 
                                                 circular_buffer_size=-1)
from keras import backend as K
import numpy as np

print("TensorFlow version: ", tf.__version__)
assert version.parse(tf.__version__).release[0] >= 2, \
    "This notebook requires TensorFlow 2.0 or above."

INFO:tensorflow:Enabled dumping callback in thread MainThread (dump root: ./logs/, tensor debug mode: FULL_HEALTH)
TensorFlow version:  2.10.0


: 

In [ ]:
data_size = 1500
# 75% of the data is for training.
train_pct = 0.75

train_size = int(data_size * train_pct)

# Create input data between -2 and 2 and shuffle it.
x = np.linspace(-2, 2, data_size)
np.random.shuffle(x)

# Generate the target values with a different linear relationship and noise.
# y = 0.8x - 1.0 + noise
y = 0.8 * x - 1.0 + np.random.normal(0, 0.1, (data_size, ))

# Split into train and test sets.
x_train, y_train = x[:train_size], y[:train_size]
x_test, y_test = x[train_size:], y[train_size:]

: 

In [ ]:
logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=logdir,
    histogram_freq=1,          # also log weight histograms
    write_graph=True,
    write_images=False
)

# Define a slightly deeper model than the original lab
model = keras.models.Sequential([
    keras.layers.Input(shape=(1,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1),
])

model.compile(
    loss='mse',
    optimizer=keras.optimizers.Adam(learning_rate=0.05),
    metrics=['mae']  # extra metric so you see more curves in TensorBoard
)

print("Training model (this should still run quickly for demo purposes)...")
training_history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    verbose=1,
    epochs=30,
    validation_data=(x_test, y_test),
    callbacks=[tensorboard_callback],
)

print("Average training loss: ", np.average(training_history.history['loss']))
print("Average validation loss: ", np.average(training_history.history['val_loss']))

Training model (this should still run quickly for demo purposes)...
Epoch 1/30
18/18 [==============================] - 14s 367ms/step - loss: 0.3024 - mae: 0.3569 - val_loss: 0.0510 - val_mae: 0.1932
Epoch 2/30
18/18 [==============================] - 5s 277ms/step - loss: 0.0220 - mae: 0.1151 - val_loss: 0.0159 - val_mae: 0.1018
Epoch 3/30
18/18 [==============================] - 5s 296ms/step - loss: 0.0118 - mae: 0.0867 - val_loss: 0.0100 - val_mae: 0.0803
Epoch 4/30
18/18 [==============================] - 6s 328ms/step - loss: 0.0100 - mae: 0.0794 - val_loss: 0.0102 - val_mae: 0.0806
Epoch 5/30
18/18 [==============================] - 5s 306ms/step - loss: 0.0095 - mae: 0.0771 - val_loss: 0.0104 - val_mae: 0.0825
Epoch 6/30
18/18 [==============================] - 4s 240ms/step - loss: 0.0095 - mae: 0.0771 - val_loss: 0.0115 - val_mae: 0.0858
Epoch 7/30
18/18 [==============================] - 4s 232ms/step - loss: 0.0109 - mae: 0.0818 - val_loss: 0.0106 - val_mae: 0.0831
Epoch 8

: 

In [ ]:

%tensorboard --logdir logs/

Reusing TensorBoard on port 6006 (pid 31208), started 0:00:23 ago. (Use '!kill 31208' to kill it.)

: 

: 